# PICKO Research · NB1 — **Breadth**: how many tools before it breaks?

**One** model is finetuned on all 40 focus tools (offered **compact**: name + description, no params — so
more names fit the encoder). Then we **only run inference**: for each tool-count `k` we offer the model
**`N_REPEATS` random subsets of `k` tools** and average the tool-selection accuracy (mean ± std). Training
is held fixed, so the curve isolates one variable — *how many tools are offered at inference* — and the
error bars remove the "unlucky tool sample" bias of testing each size once.

The 1024-token encoder truncates the offered list, so past ~20 compact tools some are never seen — that
ceiling is the result, annotated with `n_visible`.

*Run & forget:* the 40-tool model trains once to Drive and is reused; every (size, repeat) inference run
persists to `picko_out/breadth_results.json`, so a restart **skips finished runs**.

## 0 · Colab quick-start (GPU) — run & forget, restart-safe

**On Colab first: Runtime → Change runtime type → GPU (L4 recommended; T4/A100 also fine).**
This cell clones the repo, pins the exact JAX/Flax, mounts Drive, and points **both** the data (in) and
the checkpoints+results (out) at your **`MyDrive/picko/`** folder — so a runtime restart loses nothing.

**Prerequisite (one-time):** `picko_balanced.jsonl` must be in `MyDrive/picko/`. **Running locally?** This
cell is a no-op — skip to cell 1.

In [ ]:
# --- Colab bootstrap (safe to re-run; no-op locally) ---
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/picko"):
        !git clone -b hadar-work https://github.com/HadarBit/picko.git /content/picko
    %pip install -q "jax[cuda12]==0.10.2" "jaxlib==0.10.2" "flax==0.12.8"
    sys.path.insert(0, "/content/picko")
    from google.colab import drive; drive.mount("/content/drive")
    import shutil
    DRIVE = "/content/drive/MyDrive/picko"                      # <- everything lives here
    os.environ["PICKO_OUT_DIR"] = f"{DRIVE}/picko_out"          # checkpoints + results (durable)
    os.environ["PICKO_LOG"]     = f"{DRIVE}/picko_out/run.log"  # durable log across restarts
    os.makedirs(os.environ["PICKO_OUT_DIR"], exist_ok=True)
    dst = "/content/picko/data/picko_balanced.jsonl"
    if not os.path.exists(dst):
        cands = [f"{DRIVE}/picko_balanced.jsonl", "/content/drive/MyDrive/picko_balanced.jsonl"]
        src = next((c for c in cands if os.path.exists(c)), None)
        if src is None:
            have = os.listdir(DRIVE) if os.path.isdir(DRIVE) else "(MyDrive/picko not found)"
            raise FileNotFoundError(
                "picko_balanced.jsonl not found. Upload it to MyDrive/picko/. "
                f"Currently in {DRIVE}: {have}")
        os.makedirs(os.path.dirname(dst), exist_ok=True); shutil.copy(src, dst)
        print("copied data from", src)
    import jax
    print("GPU:");
    !nvidia-smi -L
    print("jax devices:", jax.devices())
    _plat = jax.devices()[0].platform
    assert _plat == "gpu", (
        f"JAX is running on '{_plat}', NOT the GPU — every finetune/eval will be ~30x slower "
        "(hours instead of minutes). FIX: Runtime > Change runtime type > GPU (L4), then "
        "Runtime > Restart session, and re-run this cell. If a GPU IS selected but this still "
        "fails, the CUDA plugin didn't load — re-run the %pip line above, then restart.")
    print("bootstrap OK · GPU active · data =", dst, "· OUT_DIR =", os.environ["PICKO_OUT_DIR"])
else:
    print("Not on Colab — running locally (CPU).")

## 1 · Setup & data overview

In [ ]:
# ensure the repo root is importable (works from notebooks/research/, Colab, etc.)
import os, sys
_here = os.path.abspath(os.getcwd())
for _ in range(6):
    if os.path.exists(os.path.join(_here, "scripts", "picko_research.py")): break
    _here = os.path.dirname(_here)
if os.path.isdir("/content/picko"): _here = "/content/picko"
if _here not in sys.path: sys.path.insert(0, _here)

from scripts.picko_research import *
import json, time
import pandas as pd, numpy as np, matplotlib.pyplot as plt
try:
    import seaborn as sns; sns.set_theme(style="whitegrid")
except Exception:
    sns = None
from tqdm.auto import tqdm

cat, tok, raw, FOCUS, OUT_DIR = load_context()
env_report(OUT_DIR)   # jax devices + is OUT_DIR durable (Drive)?

### The 40 focus tools\nOne row per tool, with its family, category and **parameter count / bucket**.

In [ ]:
display(tools_dataframe(cat, FOCUS))

### All examples for these 40 tools\nOne row per training example (query → gold tool), tagged with the gold tool's **param bucket**.

In [ ]:
ex_df = examples_dataframe(cat, raw, FOCUS)
print("examples:", ex_df.shape[0], "| per param bucket:", ex_df["param_bucket"].value_counts().to_dict())
display(ex_df.head(10))

## 2 · Configure the sweep\n`BREADTH_SIZES` = how many tools are **offered** at inference. `N_REPEATS` = how many random tool subsets we average per size (more = smoother curve, tighter error bars).

In [ ]:
BREADTH_SIZES  = [3, 5, 10, 20, 30, 40]   # tool counts OFFERED at inference (<-edit me)
N_REPEATS      = 8       # random tool subsets averaged per size -> mean +/- std (raise for a smoother curve)
CAP_PER_TOOL   = 40      # examples/tool used to build the model and the test sets
EPOCHS         = 1
EVAL_SUBSAMPLE = 60      # cap test examples per (size, repeat) for speed; None = full
MAX_GEN_LEN    = 64      # short decode: we only score the tool NAME (salvaged by regex if JSON truncates)
BATCH_SIZE     = 8       # finetune batch for the ONE model. 8 is safe on L4; raise to 16 if headroom, lower to 4 on OOM
RUN_TRAIN      = True
FORCE_RETRAIN  = False   # True = retrain the model AND recompute every inference run
print("focus tools:", len(FOCUS), "| sizes:", BREADTH_SIZES, "| repeats/size:", N_REPEATS)

## 3 · Train the ONE model (all 40 focus tools, compact)\nTrained once and reused. Every size below is inference-only on **this same model** — so differences come from the *offered* tool count, not from retraining.

In [ ]:
FOCUS40 = finetune_and_eval(cat, raw, tok, FOCUS, "breadth_focus40", OUT_DIR,
                            cap=CAP_PER_TOOL, epochs=EPOCHS, compact=True, offer_all=len(FOCUS),
                            eval_subsample=EVAL_SUBSAMPLE, run_train=RUN_TRAIN,
                            force_retrain=FORCE_RETRAIN, max_gen_len=MAX_GEN_LEN, batch_size=BATCH_SIZE)
m40, p40, tk40 = FOCUS40["bundle"]
log(f"breadth model ready · trained-on-40 selection={FOCUS40['metrics']['selection_acc']:.3f}")

## 4 · Sweep: offer k random tools, repeat, average\n*Inference only* (no training here). *Resumable:* finished (size, repeat) pairs are skipped; results persist to `OUT_DIR/breadth_results.json` after **every** run.

In [ ]:
import contextlib, io, random as _random
RES = os.path.join(OUT_DIR, "breadth_results.json")
rows = json.load(open(RES)) if (os.path.exists(RES) and not FORCE_RETRAIN) else []
done = {(r["k"], r["repeat"]) for r in rows}
if done: log(f"loaded {len(done)} finished (size,repeat) run(s) from {RES}")

t_all = time.time()
for k in BREADTH_SIZES:
    for rep in range(N_REPEATS):
        if (k, rep) in done and not FORCE_RETRAIN:
            continue
        try:
            # random subset of k focus tools for this repeat (seed=k*1000+rep -> reproducible)
            names = _random.Random(k*1000+rep).sample(list(FOCUS), min(k, len(FOCUS)))
            gset = cat.restrict_dataset(raw, names, compact=True, offer_all_max=k,
                                        cap_per_tool=CAP_PER_TOOL, seed=rep)
            _, _, test = per_tool_split(gset)
            if EVAL_SUBSAMPLE: test = test[:EVAL_SUBSAMPLE]
            with contextlib.redirect_stdout(io.StringIO()):   # silence constrained-decoder spam
                preds = predict(m40, p40, tk40, test, max_gen_len=MAX_GEN_LEN)
            met = evaluate(test, preds, family_of=family_of)
            vis = int(np.median([n_visible(e["query"], json.loads(e["tools"]), tok) for e in test]))
            rows = [r for r in rows if not (r["k"] == k and r["repeat"] == rep)] + [{
                "k": k, "repeat": rep, "selection_acc": met["selection_acc"],
                "name_f1": met["name_f1"], "parse_rate": met["parse_rate"],
                "n_visible": vis, "n_test": len(test)}]
            done.add((k, rep))
            json.dump(rows, open(RES, "w"), indent=2)   # persist each run
            log(f"k={k} rep={rep}: selection={met['selection_acc']:.3f} visible={vis}/{k}")
        except Exception as e:
            log(f"k={k} rep={rep}: FAILED ({type(e).__name__}: {e}) — skipping; re-run to resume")

log(f"ALL RUNS DONE in {time.time()-t_all:.0f}s · results={RES}")
runs = pd.DataFrame(rows)
# aggregate the repeats -> one row per size, mean +/- std
breadth = (runs.groupby("k")
           .agg(selection_mean=("selection_acc", "mean"), selection_std=("selection_acc", "std"),
                name_f1_mean=("name_f1", "mean"), parse_rate=("parse_rate", "mean"),
                n_visible=("n_visible", "median"), n_repeats=("repeat", "nunique"),
                n_test=("n_test", "sum"))
           .reset_index())
breadth["selection_std"] = breadth["selection_std"].fillna(0)
display(breadth.round(3))

## 5 · The Breadth curve (mean ± std over random tool subsets)

In [ ]:
fig, ax = plt.subplots(figsize=(8,4.5))
ax.errorbar(breadth["k"], breadth["selection_mean"], yerr=breadth["selection_std"],
            fmt="o-", color="#4C72B0", capsize=4, label="selection_acc (mean +/- std)")
# faint dots: every individual repeat, to show the spread we are averaging over
ax.scatter(runs["k"], runs["selection_acc"], s=12, color="#4C72B0", alpha=0.25, zorder=1)
wall = breadth[breadth["n_visible"] < breadth["k"]]
if len(wall):
    kw = int(wall["k"].iloc[0]); vw = int(wall["n_visible"].iloc[0])
    ax.axvline(kw, color="#C44E52", ls=":", lw=1.5)
    ax.text(kw, 0.06, f" truncation wall\n (~{vw} of {kw} tools visible)", color="#C44E52", fontsize=9, va="bottom")
ax.set_xlabel("# tools offered at inference (k)"); ax.set_ylabel("tool-selection accuracy")
ax.set_ylim(0,1.02); ax.set_title(f"Breadth: selection vs #tools offered ({int(breadth['n_repeats'].max())} random subsets/size)")
ax.legend()
plt.tight_layout(); save_fig("breadth_curve"); plt.show()

## 6 · Read-out

- **One** model (trained on all 40 tools) is probed with random subsets of `k`, so the curve reflects the
  *offered* tool count alone, not retraining. The error bars / faint dots show the spread across subsets —
  a single sample per size (the old design) could land anywhere inside that band, which is the bias we removed.
- Selection stays high for small `k` and falls as `k` grows; the red line marks where the **compact**
  offered list stops fitting the 1024-token encoder (extra tools are truncated away and can't be picked).
- **Takeaway:** one PICKO instance is bounded by the *context window it can offer*, not by what it was
  trained on — beyond the wall, a large tool set should be sharded across categorical instances.